# 20 — Milestone 3 data: entries, references and publications

Owner: Member 1. Builds the three input tables of Milestone 3 from the outputs of Milestones 1 and 2, in
`data/m3/` on Drive (`MyDrive/arxiv-project/`, **never committed**). Milestone 3 matches every BibTeX entry of
a publication (M2 `bibtex`) to the arXiv-keyed references of the same publication (M1 `references`); these
tables are the two sides of that `m × n` matrix and the list of publications.

| File | One row per | Columns (contract of the M3 plan, 21 Nov: tell the group before changing one) |
|---|---|---|
| `entries.parquet` | BibTeX entry of a publication | the 9 `bibtex` columns + `bib_source`, `position`, `n_cites`, `context`, `sections` |
| `refs.parquet` | arXiv-keyed reference of a publication | the 6 `references` columns + `ref_year`, `ref_doi`, `ref_venue`, `ref_position` |
| `pubs.parquet` | publication of the four ranges | `arxiv_id, member, n_entries, n_refs, eligible, parser_version` + `bib_source, n_cited_entries, n_refs_all, s2_status, m2_status` |

`arxiv_id` is written with a dot (`2609.32834`) in every table; a folder id (`<fid>`) is `2609-32834`.

**`entries`** (same rows and order as M2 `bibtex`)

| Column | Type | Meaning |
|---|---|---|
| `arxiv_id` … `url` | as M2 `bibtex` | `arxiv_id, citation_key, entry_type, title, authors, year, doi, journal, url`; authors `First Last`, `year` int or null |
| `bib_source` | string | where M2 read the paper's references: `bib` (a `.bib` file) or `bibitem` (a `\bibitem` list), from the last version that gave some; else `missing_bbl` / `none` |
| `position` | int32 | 0-based place of the entry in the paper's bibliography (M2 order) |
| `n_cites` | int32 | `\cite` occurrences of the key in the text, over the distinct sentences of all versions (a sentence kept in v1 and v2 is one M2 element and counts once) |
| `context` | list of strings | the citing sentences (M2 `elements`, `\cite{...}` kept, keys renamed by M2), document order, v1 first, at most `CONTEXT_MAX` = 10 |
| `sections` | list of strings | the titles of the nearest `section` / `chapter` / `abstract` above each citing sentence (walking up `parent_id`), distinct, in order |

**`refs`** (only the references that S2 gives with an arXiv ID; a reference S2 lists twice is kept once)

| Column | Type | Meaning |
|---|---|---|
| `arxiv_id, ref_arxiv_id, ref_title, ref_authors, ref_submission_date, ref_s2_id` | as M1 `references` | the rows of `references_rows()` of `03`; `ref_arxiv_id` through `normalize_ref_arxiv_id` (old IDs such as `hep-th/9901001` kept) |
| `ref_year` | int32 | S2 year |
| `ref_doi` | string | S2 `externalIds.DOI` |
| `ref_venue` | string | S2 venue |
| `ref_position` | int32 | 0-based place in the paper's whole S2 list (arXiv or not) |

**`pubs`**: `member` from `MEMBER_RANGES`; `n_entries` / `n_refs` = rows in `entries` / `refs`; `eligible` =
`n_entries ≥ 1` and `n_refs ≥ 1` (a paper the matcher can work on); `parser_version` = the M2 parser of the
paper (from `state/m2/done.jsonl`, see section 2). Extra columns, to explain who is not eligible:
`bib_source` (as above), `n_cited_entries` (entries with `n_cites > 0`), `n_refs_all` (all S2 references),
`s2_status` (`ok` / `not_found` / `error` / `missing` / `unreadable`), `m2_status` (`ok` if a version was
parsed; else `empty_source` / `no_root` / `error`, or `missing` / `unreadable` for the `data/parsed/` file).

**Inputs.** The papers of `state/discovery_manifest.json` (the list of `15`); for each one
`data/parsed/<fid>.json` (written by `15`: bibtex rows, the tree of every version with element IDs, the
elements) and `data/raw/<fid>/references.json` (`03`). One pass, paper by paper: the 10 M `elements` rows are
never in memory at once. **Parser lock:** the labels key on `(arxiv_id, citation_key)`, and parsing again can
rename keys, so Milestone 3 uses the parser of 22 Nov, `M3_PARSER_VERSION`.

| Section | Functions |
|---|---|
| 0. Setup | Colab clone, settings, `13` (which loads `00`, `10`–`12`), `H12`, `H03` |
| 1. Contract | `ENTRY_COLUMNS`, `REF_COLUMNS`, `PUB_COLUMNS`, `M3_SCHEMAS` |
| 2. Papers and members | `MEMBER_RANGES`, `member_of`, `m3_papers`, `spread_sample`, `m2_parser_versions` |
| 3. One paper | `citation_contexts`, `paper_bib_source`, `entry_rows`, `ref_rows`, `pub_row`, `paper_rows` |
| 4. All papers | `rows_for_papers`, `frames_of`, `build_m3_data` (resumable), `load_m3` |
| 5. Checks and summary | `check_m3_tables`, `m3_summary`, `print_m3_summary` |
| 6. Tests | `M3DataTests` |
| 7. Run | `BUILD_M3 = True` on Colab |

`21_labeling`, `22_features` and `25_run_m3` run this notebook with `%run -i ./20_m3_data.ipynb` (it builds
nothing then): they get `H12` and `H03` (helpers of `12` and `03`), `M3_DIR`, `LABEL_DIR`, `load_m3`,
`MEMBER_RANGES` and `member_of`. Output of a build: the three files and `logs/m3_data_summary.json`.

## 0. Setup

**On Colab** (CPU runtime): open this notebook, set `BUILD_M3 = True` below (and `PREPARE_SHEETS = True` to
make the manual label sheets right after, section 7), *Run all*. The first cell clones
the branch into `/content` and works from `src/`, like `15`. The tests of section 6 run first (offline, plus
20 real papers of `data/parsed/`); the build reads about 40,000 files on Drive with `THREADS` threads and
writes one part per `BATCH` papers, so after a disconnect *Run all* again goes on from the last finished
batch. A build that finds `data/m3/` already made from the same papers, parser and `M3_DATA_VERSION` does
nothing (`FORCE_REBUILD = True` builds again).

Another notebook can set `M3_OVERRIDES = {...}` before `%run -i` to change any value of the settings cell.

In [ ]:
# On Colab, when this notebook is opened on its own: clone or update the branch, install the packages and work
# from src/ (the notebooks use relative paths). When 21, 22 or 25 run it with %run -i, the working folder is
# already src/ and nothing happens here.
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB and not Path("./00_config.ipynb").is_file():
    REPO = Path("/content/intro-ds-arxiv-project")
    BRANCH = "m3-labeling"   # the Milestone 3 branch: 13, this notebook and 21-23
    URL = "https://github.com/thinhelzan842/intro-ds-arxiv-project.git"
    if not (REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)], check=True)
    else:   # explicit refspec: also works in a clone of another single branch
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin",
                        f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "src/requirements.txt")], check=True)
    os.chdir(REPO / "src")
    print("Code in use:", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"],
                                         capture_output=True, text=True).stdout.strip())

if "RUN_TEST" not in globals():
    RUN_TEST = True        # tests of section 6 (offline, plus 20 real papers when data/parsed/ exists)
BUILD_M3 = False           # True on Colab: read every paper and write data/m3/ (section 7)
PREPARE_SHEETS = False     # True with BUILD_M3: then 21's prepare_manual_sheets() (labels/selection.csv, labels/todo/)
FORCE_REBUILD = False      # True: build again even if data/m3/ holds the same papers, parser and M3_DATA_VERSION
TEST_PAPERS = None         # N: build from N papers only (spread over the four ranges), into data/m3_test/
THREADS = 16               # parallel file reads (Drive reads are the bottleneck, not the CPU)
BATCH = 500                # papers per part: after a disconnect, one batch at most is read again
READ_TIMEOUT = 600         # seconds to read one batch before giving up (Drive may stop answering)
FLUSH_DRIVE_AT_END = True  # Colab, after a build: flush and unmount Drive, so every file is saved
globals().update(globals().get("M3_OVERRIDES", {}))

In [ ]:
_RUN_TEST_20 = RUN_TEST
RUN_TEST = False   # the tests of 13 and of the notebooks it loads are not run from here
if "parse_paper" not in globals() or "rows_from_parsed" not in globals():
    get_ipython().run_line_magic("run", "-i ./13_dedup_export.ipynb")   # runs 00_config and loads 10-12
RUN_TEST = _RUN_TEST_20

import hashlib
import shutil
import tempfile
import unittest
import warnings
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import pyarrow as pa

try:
    from tqdm.auto import tqdm as _tqdm
except ImportError:
    _tqdm = None

def load_functions(notebook, names, namespace):
    """Only the named top-level functions of ./<notebook>.ipynb, run in `namespace` (returned)."""
    names, found = set(names), set()
    for cell in json.loads(Path(f"./{notebook}.ipynb").read_text(encoding="utf-8"))["cells"]:
        if cell["cell_type"] != "code" or not any(f"def {n}(" in "".join(cell["source"]) for n in names):
            continue
        for node in ast.parse("".join(cell["source"])).body:
            if isinstance(node, ast.FunctionDef) and node.name in names:
                exec(compile(ast.Module([node], []), f"<{notebook}>", "exec"), namespace)
                found.add(node.name)
    if found != names:
        raise NameError(f"{notebook} has no {sorted(names - found)}")
    return namespace


# Helpers of 12 (norm_title, latex_to_text, surname_key, norm_doi, find_arxiv_id, parse_year, cited_keys, ...)
# and of 03, each in its own namespace. 12 through load_definitions of 13 (its definitions only); from 03 only
# these functions, since the rest of 03 sets up the S2 client (its helpers split_version, NEW_ID_RE, OLD_ID_RE,
# to_folder_id, read_json, DATA_DIR come from 00_config).
H12 = TEAM["12_bibtex"] if "12_bibtex" in TEAM else load_definitions("12_bibtex", {**globals(), "RUN_TEST": False})
H03 = load_functions("03_references", ("normalize_ref_arxiv_id", "references_path", "_saved_references",
                                       "references_rows"), {**globals()})

M3_DATA_VERSION = "20.3"                             # bump when the rows built here change: data/m3/ is built again
M3_PARSER_VERSION = "13.9|10:team|11:team|12:team"   # the M2 parser locked for Milestone 3 (13.9: 27 Nov)
# Manual labels made on these versions stay valid: 13.9 changed only the 190 REVTeX papers (none is labelled
# by hand) and 15 checked that the 20 manual papers give the same file under 13.9 (CARRY_CHECK)
M3_LABEL_PARSER_VERSIONS = (M3_PARSER_VERSION, "13.8|10:team|11:team|12:team")
if PARSER_VERSION != M3_PARSER_VERSION:
    warnings.warn(f"13 now gives parser {PARSER_VERSION}, Milestone 3 is locked to {M3_PARSER_VERSION}: "
                  "data/parsed/ is read as it is; running 15 again would change the citation keys of the labels")

PARSED_DIR = DATA_DIR.parent / "parsed"         # data/parsed/<fid>.json, written by 15
M3_DIR = DATA_DIR.parent / "m3"                 # data/m3/: entries, refs, pubs (here); pairs, predictions (22, 23)
LABEL_DIR = PROJECT_ROOT / "labels"             # labels/manual/, labels/auto/, labels/split.json (21)
M2_INDEX = STATE_DIR / "m2" / "done.jsonl"      # 15's resume index: the parser version of every paper
M3_SUMMARY_PATH = LOG_DIR / "m3_data_summary.json"

## 1. Contract: columns and types

The column lists of the plan, and the Arrow types of the Parquet files. The 9 `bibtex` columns keep the
types of M2's `bibtex.parquet` (`entry_type` dictionary-encoded, `authors` a list of strings, `year` int32).
The lists `context`, `sections`, `authors`, `ref_authors` are never null (an empty list instead).

In [ ]:
ENTRY_COLUMNS = tuple(BIBTEX_COLUMNS) + ("bib_source", "position", "n_cites", "context", "sections")
REF_COLUMNS = ("arxiv_id", "ref_arxiv_id", "ref_title", "ref_authors", "ref_submission_date", "ref_s2_id",
               "ref_year", "ref_doi", "ref_venue", "ref_position")
PUB_COLUMNS = ("arxiv_id", "member", "n_entries", "n_refs", "eligible", "parser_version",
               "bib_source", "n_cited_entries", "n_refs_all", "s2_status", "m2_status")

_STR, _INT, _LIST = pa.string(), pa.int32(), pa.list_(pa.string())
M3_SCHEMAS = {
    "entries": pa.schema([
        pa.field("arxiv_id", _STR, nullable=False), pa.field("citation_key", _STR, nullable=False),
        pa.field("entry_type", pa.dictionary(pa.int32(), pa.string())), pa.field("title", _STR),
        pa.field("authors", _LIST), pa.field("year", _INT), pa.field("doi", _STR), pa.field("journal", _STR),
        pa.field("url", _STR), pa.field("bib_source", _STR), pa.field("position", _INT, nullable=False),
        pa.field("n_cites", _INT, nullable=False), pa.field("context", _LIST), pa.field("sections", _LIST)]),
    "refs": pa.schema([
        pa.field("arxiv_id", _STR, nullable=False), pa.field("ref_arxiv_id", _STR, nullable=False),
        pa.field("ref_title", _STR), pa.field("ref_authors", _LIST), pa.field("ref_submission_date", _STR),
        pa.field("ref_s2_id", _STR), pa.field("ref_year", _INT), pa.field("ref_doi", _STR),
        pa.field("ref_venue", _STR), pa.field("ref_position", _INT, nullable=False)]),
    "pubs": pa.schema([
        pa.field("arxiv_id", _STR, nullable=False), pa.field("member", _STR),
        pa.field("n_entries", _INT, nullable=False), pa.field("n_refs", _INT, nullable=False),
        pa.field("eligible", pa.bool_(), nullable=False), pa.field("parser_version", _STR),
        pa.field("bib_source", _STR), pa.field("n_cited_entries", _INT, nullable=False),
        pa.field("n_refs_all", _INT, nullable=False), pa.field("s2_status", _STR), pa.field("m2_status", _STR)]),
}
M3_TABLES = tuple(M3_SCHEMAS)
assert [tuple(M3_SCHEMAS[t].names) for t in M3_TABLES] == [ENTRY_COLUMNS, REF_COLUMNS, PUB_COLUMNS]

CONTEXT_MAX = 10                                         # citing sentences kept per entry (n_cites counts them all)
SECTION_KINDS = frozenset({"chapter", "section", "abstract"})   # what `sections` names
MANUAL_POOL = {"min_entries": 20, "max_entries": 60, "min_refs": 10}   # papers to label by hand (plan)
SPLIT_POOL = {"min_entries": 20, "min_refs": 10}                       # papers split.json may draw from (plan)

## 2. Papers and members

- `m3_papers()`: the papers with at least one version in the discovery manifest, the same list as `15`
  (without a manifest, the paper folders of `data/raw/`), in ID order. `spread_sample(papers, n)` takes about
  `n` of them spread over the four ranges (tests, `TEST_PAPERS`).
- `MEMBER_RANGES`: the range of each member, from the group's registration sheet; `member_of(arxiv_id)`
  gives the member whose range holds an ID (`None` outside the four ranges).
- `m2_parser_versions()`: the parser version of every paper, from `15`'s resume index
  `state/m2/done.jsonl`, not from the `data/parsed/` file. The papers that `CARRY_OVER` kept on 22 Nov are
  `13.8` in the index while their file still says `13.7` (13.8 cannot change them; `15` checked it on 300
  papers parsed again). The same on 27 Nov for 13.8 → 13.9, which parsed again only the 190 papers with
  REVTeX `\bibitem`s: the others are `13.9` in the index and keep their file.

In [ ]:
MEMBER_RANGES = {   # member -> (first, last) arXiv ID: the course's registration sheet, group 10
    "TV1": ("2603.20580", "2603.25579"),   # La Anh Hào
    "TV2": ("2609.32834", "2609.37833"),   # Võ Nguyễn Vân Anh
    "TV3": ("2605.17338", "2605.22337"),   # Lưu Minh Quân
    "TV4": ("2605.07338", "2605.12337"),   # Phạm Tấn Nhật Thịnh
}
assert sorted(MEMBER_RANGES.values()) == sorted(map(tuple, ID_RANGES)), "MEMBER_RANGES must be the ranges of 00"


def _id_key(arxiv_id):
    """'2609.32834' (or '2609.32834v2') -> ('2609', 32834); None for any other form."""
    m = NEW_ID_RE.match(split_version(str(arxiv_id))[0])
    return (m.group(1), int(m.group(2))) if m else None


def _paper_order(paper):
    return _id_key(paper[0]) or (str(paper[0]), 0)


def member_of(arxiv_id):
    """The member whose range holds the ID, or None."""
    key = _id_key(arxiv_id)
    if key is None:
        return None
    for member, (first, last) in MEMBER_RANGES.items():
        if _id_key(first) <= key <= _id_key(last):
            return member
    return None


def spread_sample(papers, n):
    """About n of the [(arxiv_id, folder_id)] spread evenly over the members' ranges, in ID order."""
    groups = {}
    for paper in papers:
        groups.setdefault(member_of(paper[0]), []).append(paper)
    per = max(1, n // max(len(groups), 1))
    picked = [p for group in groups.values() for p in group[::max(1, len(group) // per)][:per]]
    return sorted(picked, key=_paper_order)


def m3_papers(test_papers=None):
    """[(arxiv_id, folder_id)] of every paper with at least one version, in ID order (the list of 15)."""
    manifest = read_json(STATE_DIR / "discovery_manifest.json")
    if manifest:
        records = manifest["papers"] if isinstance(manifest, dict) else manifest
        ids = {split_version(p["id"])[0] for p in records if (p.get("versions") or 0) > 0}
    else:
        ids = {name.replace("-", ".") for name in os.listdir(DATA_DIR) if re.fullmatch(r"\d{4}-\d{4,5}", name)}
    papers = sorted(((i, to_folder_id(i)) for i in ids), key=_paper_order)
    return spread_sample(papers, test_papers) if test_papers else papers


def m2_parser_versions(path=None):
    """{folder_id: parser_version} of 15's resume index (the last line of a paper wins); {} without one."""
    path = Path(path or M2_INDEX)
    versions = {}
    if path.is_file():
        with open(path, encoding="utf-8") as stream:
            for line in stream:
                try:
                    record = json.loads(line)
                except ValueError:   # a line cut by a disconnect
                    continue
                if record.get("folder_id"):
                    versions[record["folder_id"]] = record.get("parser_version")
    return versions

## 3. One paper

`data/parsed/<fid>.json` stores each version's tree as nested lists `[element_id, child, child, ...]` and
the elements column by column (`element_id, arxiv_id, kind, content`), see `to_parsed_json` in `13`.

- `citation_contexts(doc)` walks the tree of every version in document order (v1 first), keeping the
  nearest `section` / `chapter` / `abstract` above each node. A leaf (`sentence`, `formula`, `figure`) whose
  text holds a cite command is read with `cited_keys` of `12` (`\cite`, `\citep[..][..]{a,b}`, `\citet`,
  `\cites`, ...). M2 gives one element to identical leaves, in any version and under any parent, so a leaf
  is counted **once** per paper (`n_cites`, `context`), while each of its places adds its section title.
  Keys cited in the text but missing from the bibliography are counted in the build statistics.
- `entry_rows(arxiv_id, doc)`: the `bibtex` rows of the file in their order (`position`), with the context
  of their key; a repeated key keeps its first row (none expected: M2 dedups the keys).
- `ref_rows(arxiv_id, refs_doc)`: the rows of `references_rows` of `03` for this paper (references with an
  arXiv ID, `ok` files only) plus year, DOI, venue and the place in the whole S2 list; an `ref_arxiv_id` that
  S2 lists twice keeps its first row (the matrix needs one column per reference).
- `pub_row`, `paper_rows`: the `pubs` row, and the three for one paper. A missing or unreadable file (torn by
  a Drive disconnect) gives no rows and is visible in `m2_status` / `s2_status`.

In [ ]:
def _read_doc(path):
    """(document, None), or (None, "missing") / (None, "unreadable") (a file torn by a Drive disconnect)."""
    path = Path(path)
    if not path.is_file():
        return None, "missing"
    try:
        return read_json(path), None
    except (OSError, ValueError):
        return None, "unreadable"


def _int_or_none(value):
    if value is None or isinstance(value, bool):
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def _str_list(value):
    if not value:
        return []
    if isinstance(value, str):
        return [value]
    return [str(v) for v in value if v is not None]


def citation_contexts(doc, max_sentences=CONTEXT_MAX):
    """{citation_key: {"context": [...], "n_cites": int, "sections": [...]}} of one parsed.json document."""
    elements = doc.get("elements") or {}
    info = dict(zip(elements.get("element_id") or [],
                    zip(elements.get("kind") or [], elements.get("content") or [])))
    found, seen = {}, set()
    for version in sorted(doc.get("trees") or {}, key=int):
        stack = [(doc["trees"][version], "")]
        while stack:
            node, section = stack.pop()
            kind, content = info.get(node[0], ("", ""))
            content = content or ""
            if kind in SECTION_KINDS:
                section = content or kind
            if len(node) > 1:
                stack.extend((child, section) for child in reversed(node[1:]))
                continue
            if kind not in LEAF_KINDS or "cite" not in content.lower():
                continue
            first = node[0] not in seen
            seen.add(node[0])
            for key, count in H12["cited_keys"](content).items():
                slot = found.setdefault(key, {"context": [], "n_cites": 0, "sections": []})
                if first:
                    slot["n_cites"] += count
                    if len(slot["context"]) < max_sentences:
                        slot["context"].append(content)
                if section and section not in slot["sections"]:
                    slot["sections"].append(section)
    return found


def paper_bib_source(doc):
    """bib / bibitem of the last version that gave references; else the last version's value; 'none' without one."""
    versions = (doc or {}).get("versions") or []
    for record in reversed(versions):
        if record.get("status") == "ok" and record.get("bib_source") in ("bib", "bibitem"):
            return record["bib_source"]
    return (versions[-1].get("bib_source") or "none") if versions else "none"


def entry_rows(arxiv_id, doc):
    """(rows of entries.parquet, stats) of one parsed.json document, in its bibtex order."""
    stats, keys, bib = Counter(), set(), []
    for row in (rows_from_parsed(doc, "bibtex") if doc.get("bibtex") else []):
        key = str(row.get("citation_key") or "").strip()
        if not key or key in keys:
            stats["entry_duplicate_keys"] += 1
            continue
        keys.add(key)
        bib.append({**row, "citation_key": key})
    contexts = citation_contexts(doc)
    stats["cited_keys_not_in_bibtex"] += len(set(contexts) - keys - {"*"})
    source = paper_bib_source(doc)
    rows = []
    for position, row in enumerate(bib):
        found = contexts.get(row["citation_key"]) or {"context": [], "n_cites": 0, "sections": []}
        rows.append({
            "arxiv_id": arxiv_id, "citation_key": row["citation_key"], "entry_type": row.get("entry_type") or "misc",
            "title": row.get("title"), "authors": _str_list(row.get("authors")), "year": _int_or_none(row.get("year")),
            "doi": row.get("doi"), "journal": row.get("journal"), "url": row.get("url"), "bib_source": source,
            "position": position, "n_cites": found["n_cites"], "context": found["context"],
            "sections": found["sections"]})
    return rows, stats


def ref_rows(arxiv_id, refs_doc):
    """(rows of refs.parquet, duplicates dropped) of one references.json document."""
    if not refs_doc or refs_doc.get("status") != "ok":
        return [], 0
    rows, seen, dropped = [], set(), 0
    for position, ref in enumerate(refs_doc.get("references") or []):
        ref_id = H03["normalize_ref_arxiv_id"](ref.get("arxiv_id"))[0]
        if not ref_id:
            continue
        if ref_id in seen:
            dropped += 1
            continue
        seen.add(ref_id)
        date = ref.get("submission_date")
        rows.append({
            "arxiv_id": arxiv_id, "ref_arxiv_id": ref_id, "ref_title": ref.get("title"),
            "ref_authors": _str_list(ref.get("authors")), "ref_submission_date": None if date is None else str(date),
            "ref_s2_id": ref.get("s2_id"), "ref_year": _int_or_none(ref.get("year")),
            "ref_doi": (ref.get("external_ids") or {}).get("DOI") or None, "ref_venue": ref.get("venue") or None,
            "ref_position": position})
    return rows, dropped


def pub_row(arxiv_id, doc, refs_doc, entries, refs, parser_version=None, m2_problem=None, s2_problem=None):
    """The pubs.parquet row of one paper (`parser_version`: its version in 15's index, if known)."""
    versions = (doc or {}).get("versions") or []
    if doc is None:
        m2_status = m2_problem or "missing"
    elif any(v.get("status") == "ok" for v in versions):
        m2_status = "ok"
    else:
        m2_status = (versions[-1].get("status") or "error") if versions else "no_version"
    s2_status = (refs_doc or {}).get("status") or s2_problem or "missing"
    return {
        "arxiv_id": arxiv_id, "member": member_of(arxiv_id), "n_entries": len(entries), "n_refs": len(refs),
        "eligible": bool(entries) and bool(refs),
        "parser_version": parser_version or (doc or {}).get("parser_version"),
        "bib_source": paper_bib_source(doc), "n_cited_entries": sum(e["n_cites"] > 0 for e in entries),
        "n_refs_all": len(refs_doc.get("references") or []) if s2_status == "ok" else 0,
        "s2_status": s2_status, "m2_status": m2_status}


def paper_rows(arxiv_id, folder_id, parsed_dir=None, raw_dir=None, parser_version=None):
    """(entries rows, refs rows, pubs row, stats) of one paper."""
    doc, m2_problem = _read_doc(Path(parsed_dir or PARSED_DIR) / f"{folder_id}.json")
    refs_doc, s2_problem = _read_doc(Path(raw_dir or DATA_DIR) / folder_id / "references.json")
    entries, stats = entry_rows(arxiv_id, doc) if doc else ([], Counter())
    refs, dropped = ref_rows(arxiv_id, refs_doc)
    stats["ref_duplicates_dropped"] += dropped
    pub = pub_row(arxiv_id, doc, refs_doc, entries, refs, parser_version, m2_problem, s2_problem)
    return entries, refs, pub, stats

## 4. All papers

- `rows_for_papers(papers)`: the rows of a list of papers, the files read by `THREADS` threads (Drive
  reads wait on the network, so threads help); `frames_of(...)` turns them into DataFrames with the
  contract columns. Both work without Parquet (the tests, a machine where `pyarrow.parquet` is blocked).
- `build_m3_data()`: every paper, `BATCH` at a time. Each batch is written to
  `data/m3/_parts/<signature>/` (written locally, then copied and renamed, so a disconnect never leaves half
  a file; the `pubs` part is written last and marks a finished batch), then the parts are merged into the
  three files, checked (`check_m3_tables`), summarized into `logs/m3_data_summary.json` and removed. The
  signature hashes the paper list, `BATCH`, the parser and `M3_DATA_VERSION`; `data/m3/_build.json` records
  the last build, so a second call with the same signature does nothing. It stops before reading anything
  when `state/m2/done.jsonl` holds papers parsed with another parser than `M3_PARSER_VERSION`.
- `load_m3(table)`: a table as a DataFrame (`entry_type` comes back as a pandas category).

In [ ]:
def rows_for_papers(papers, parsed_dir=None, raw_dir=None, versions=None, threads=THREADS, timeout=READ_TIMEOUT):
    """(entries, refs, pubs, stats) of [(arxiv_id, folder_id)], rows in the order of `papers`."""
    versions = m2_parser_versions() if versions is None else versions
    pool = ThreadPoolExecutor(max(1, threads))
    try:
        results = list(pool.map(lambda p: paper_rows(p[0], p[1], parsed_dir, raw_dir, versions.get(p[1])),
                                papers, timeout=timeout))
    except TimeoutError:
        raise RuntimeError(f"reading {len(papers)} papers took more than {timeout} s: is Google Drive still "
                           "mounted? Run again: the finished batches are kept.") from None
    finally:
        pool.shutdown(wait=False, cancel_futures=True)   # a thread stuck on Drive is left behind
    entries, refs, pubs, stats = [], [], [], Counter()
    for e, r, p, s in results:
        entries += e
        refs += r
        pubs.append(p)
        stats.update(s)
    return entries, refs, pubs, stats


def frames_of(entries, refs, pubs):
    """(entries, refs, pubs) rows -> three DataFrames with the contract columns."""
    return tuple(pd.DataFrame(rows, columns=list(M3_SCHEMAS[t].names))
                 for t, rows in zip(M3_TABLES, (entries, refs, pubs)))


def _to_table(rows, schema):
    return pa.Table.from_pydict({name: [row[name] for row in rows] for name in schema.names}, schema=schema)


def _write_parquet(table, path):
    """Written on the local disk, then copied next to `path` and renamed: never half a file on Drive."""
    import pyarrow.parquet as pq
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with tempfile.TemporaryDirectory(prefix="m3-") as local:
        pq.write_table(table, Path(local) / "part.parquet", compression="zstd")
        shutil.copyfile(Path(local) / "part.parquet", tmp)
    os.replace(tmp, path)


def _signature(papers, batch):
    text = json.dumps([M3_DATA_VERSION, M3_PARSER_VERSION, batch, [folder for _, folder in papers]])
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:12]


def build_m3_data(papers=None, out_dir=None, batch=BATCH, threads=THREADS, force=None, allow_other_parser=False,
                  parsed_dir=None, raw_dir=None, index=None):
    """Write entries/refs/pubs.parquet of `papers` (default m3_papers()) to out_dir (default data/m3/).

    Returns the build record (signature, rows, stats, problems, summary), also written to out_dir/_build.json.
    """
    import pyarrow.compute as pc   # here: on some Windows machines their DLLs are blocked, and 21/22 only need H12/H03
    import pyarrow.parquet as pq

    force = FORCE_REBUILD if force is None else force
    papers = m3_papers() if papers is None else list(papers)
    out_dir, parsed_dir = Path(out_dir or M3_DIR), Path(parsed_dir or PARSED_DIR)
    signature = _signature(papers, batch)
    stamp = read_json(out_dir / "_build.json") or {}
    if not force and stamp.get("signature") == signature and all((out_dir / f"{t}.parquet").is_file() for t in M3_TABLES):
        print(f"[20] {out_dir} is up to date: built {stamp.get('built_at')}, rows {stamp.get('rows')} "
              f"(FORCE_REBUILD = True builds again)")
        return stamp
    if not parsed_dir.is_dir():
        raise FileNotFoundError(f"{parsed_dir} does not exist: 15 writes it (on Colab, is Drive mounted?)")
    versions = m2_parser_versions(index)
    # None in the index: 15 found the file unreadable and parses the paper again next session (not another parser)
    other = Counter(versions[f] for _, f in papers if versions.get(f) not in (None, M3_PARSER_VERSION))
    if other:
        message = (f"{sum(other.values()):,} papers are not {M3_PARSER_VERSION} in the M2 index: "
                   f"{dict(other.most_common(3))}")
        if not allow_other_parser:
            raise RuntimeError(message + " (finish the M2 run of 15 first, or allow_other_parser=True)")
        print("[20] warning:", message)
    unindexed = sum(versions.get(f) is None for _, f in papers)
    print(f"[20] {len(papers):,} papers -> {out_dir} | parser {M3_PARSER_VERSION} | "
          + (f"{unindexed:,} not in the M2 index" if versions else "no M2 index: the version of each file is used"))

    t0 = time.time()
    parts = out_dir / "_parts" / signature
    parts.mkdir(parents=True, exist_ok=True)
    starts = list(range(0, len(papers), batch))
    part = lambda table, number: parts / f"{table}-{number:05d}.parquet"   # noqa: E731
    bar = _tqdm(total=len(papers), unit="paper", desc="M3 data", dynamic_ncols=True) if _tqdm else None
    try:
        for number, start in enumerate(starts):
            chunk = papers[start:start + batch]
            if not part("pubs", number).is_file():   # written last: a finished batch
                entries, refs, pubs, stats = rows_for_papers(chunk, parsed_dir, raw_dir, versions, threads)
                _write_parquet(_to_table(entries, M3_SCHEMAS["entries"]), part("entries", number))
                _write_parquet(_to_table(refs, M3_SCHEMAS["refs"]), part("refs", number))
                write_json(parts / f"stats-{number:05d}.json", dict(stats))
                _write_parquet(_to_table(pubs, M3_SCHEMAS["pubs"]), part("pubs", number))
            if bar is not None:
                bar.update(len(chunk))
    finally:
        if bar is not None:
            bar.close()

    rows, local = {}, Path(tempfile.mkdtemp(prefix="m3-tables-"))
    try:
        for table, schema in M3_SCHEMAS.items():
            rows[table] = 0
            with pq.ParquetWriter(local / f"{table}.parquet", schema, compression="zstd") as writer:
                for number in range(len(starts)):
                    piece = pq.read_table(part(table, number)).cast(schema)
                    writer.write_table(piece)
                    rows[table] += piece.num_rows
        out_dir.mkdir(parents=True, exist_ok=True)
        for table in M3_SCHEMAS:
            target = out_dir / f"{table}.parquet"
            shutil.copyfile(local / f"{table}.parquet", target.with_name(target.name + ".tmp"))
            os.replace(target.with_name(target.name + ".tmp"), target)
    finally:
        shutil.rmtree(local, ignore_errors=True)
    stats = Counter()
    for number in range(len(starts)):
        stats.update(read_json(parts / f"stats-{number:05d}.json") or {})

    # Checks on what was written (entries without the text columns: the length of `context` is enough)
    read = pq.read_table(out_dir / "entries.parquet", columns=["arxiv_id", "citation_key", "position", "n_cites", "context"])
    entries = read.select(["arxiv_id", "citation_key", "position", "n_cites"]).to_pandas()
    entries["n_context"] = pc.list_value_length(read["context"]).fill_null(0).to_numpy()
    refs = pd.read_parquet(out_dir / "refs.parquet", columns=["arxiv_id", "ref_arxiv_id"])
    pubs = pd.read_parquet(out_dir / "pubs.parquet")
    problems = check_m3_tables(entries, refs, pubs, full=False,
                               parser_version=None if allow_other_parser else M3_PARSER_VERSION)
    record = {"signature": signature, "built_at": to_iso(datetime.now(timezone.utc)),
              "seconds": round(time.time() - t0, 1), "parser_version": M3_PARSER_VERSION,
              "data_version": M3_DATA_VERSION, "papers": len(papers), "rows": rows, "stats": dict(stats),
              "problems": problems[:50], "summary": m3_summary(pubs, stats)}
    write_json(M3_SUMMARY_PATH if out_dir == M3_DIR else out_dir / "summary.json", record)
    if problems:
        print("\n".join(f"  - {p}" for p in problems[:20]))
        raise RuntimeError(f"[20] {len(problems)} problems in {out_dir}: not marked as built, the parts are kept")
    write_json(out_dir / "_build.json", record)
    shutil.rmtree(out_dir / "_parts", ignore_errors=True)
    unreadable = int(pubs["m2_status"].eq("unreadable").sum() + pubs["s2_status"].eq("unreadable").sum())
    print(f"[20] wrote {out_dir} in {record['seconds']:.0f} s: "
          + ", ".join(f"{t} {n:,} rows" for t, n in rows.items())
          + (f" | warning: {unreadable} unreadable files (run 15 / 07 again for them)" if unreadable else ""))
    return record


def load_m3(table=None, out_dir=None, columns=None):
    """data/m3/<table>.parquet as a DataFrame; without `table`, {name: DataFrame} of the three."""
    out_dir = Path(out_dir or M3_DIR)
    if table:
        return pd.read_parquet(out_dir / f"{table}.parquet", columns=columns)
    return {t: pd.read_parquet(out_dir / f"{t}.parquet") for t in M3_TABLES}

## 5. Checks and summary

`check_m3_tables(entries, refs, pubs)` lists the broken invariants (empty when all hold):

- the contract columns, in order (`full=True`); `arxiv_id` written like `2609.32834`; every paper in
  `pubs` once and inside `MEMBER_RANGES`; no entry or reference of a paper missing from `pubs`;
- `(arxiv_id, citation_key)` and `(arxiv_id, ref_arxiv_id)` unique and non-empty; `ref_arxiv_id` already in
  the form of `normalize_ref_arxiv_id`;
- `n_entries`, `n_refs` equal to the rows of each paper; `eligible` = both ≥ 1;
- `position` is `0 … m−1` in each paper; `n_cites > 0` exactly when `context` is not empty, and
  `context` holds at most `CONTEXT_MAX` sentences;
- every paper whose `data/parsed/` file was read has `parser_version == M3_PARSER_VERSION`.

`m3_summary(pubs)` gives the numbers for the group, the README and the poster: eligible papers (overall
and per member), entries and arXiv references, the size of the matching matrices, and the pools of the
plan: papers to label by hand (20–60 entries, ≥ 10 arXiv references) and papers the split may use (≥ 20
entries, ≥ 10 arXiv references), per member and `bib_source`.

In [ ]:
def check_m3_tables(entries, refs, pubs, full=True, parser_version=M3_PARSER_VERSION):
    """Broken invariants of the three tables (an empty list when all hold).

    full=False: the frames hold only the columns checked here (`entries` may carry `n_context` instead of
    `context`), so the column lists are not compared with the contract. parser_version=None: not checked.
    """
    problems = []
    if full:
        for name, frame, columns in (("entries", entries, ENTRY_COLUMNS), ("refs", refs, REF_COLUMNS),
                                     ("pubs", pubs, PUB_COLUMNS)):
            if tuple(frame.columns) != tuple(columns):
                problems.append(f"{name}: columns {list(frame.columns)}, the contract is {list(columns)}")
    pub_ids = pubs["arxiv_id"].astype(str)
    bad = ~pub_ids.str.fullmatch(r"\d{4}\.\d{4,5}")
    if bad.any():
        problems.append(f"pubs: {int(bad.sum())} arxiv_id not written like 2609.32834, e.g. {pub_ids[bad].iloc[0]!r}")
    if pub_ids.duplicated().any():
        problems.append(f"pubs: {int(pub_ids.duplicated().sum())} duplicate arxiv_id")
    if pubs["member"].isna().any():
        problems.append(f"pubs: {int(pubs['member'].isna().sum())} papers outside MEMBER_RANGES")
    known = set(pub_ids)
    for name, frame, key in (("entries", entries, "citation_key"), ("refs", refs, "ref_arxiv_id")):
        ids = frame["arxiv_id"].astype(str)
        if (~ids.isin(known)).any():
            problems.append(f"{name}: {int((~ids.isin(known)).sum())} rows of papers that are not in pubs")
        empty = frame[key].isna() | (frame[key].astype(str).str.strip() == "")
        if empty.any():
            problems.append(f"{name}: {int(empty.sum())} empty {key}")
        dup = frame.duplicated(["arxiv_id", key])
        if dup.any():
            problems.append(f"{name}: {int(dup.sum())} duplicate (arxiv_id, {key}), "
                            f"e.g. {frame.loc[dup, ['arxiv_id', key]].head(3).values.tolist()}")
    bad_refs = [r for r in refs["ref_arxiv_id"].dropna() if H03["normalize_ref_arxiv_id"](r)[0] != r]
    if bad_refs:
        problems.append(f"refs: {len(bad_refs)} ref_arxiv_id not normalized, e.g. {bad_refs[:3]}")
    p = pubs.assign(arxiv_id=pub_ids).drop_duplicates("arxiv_id").set_index("arxiv_id")
    for name, frame, column in (("entries", entries, "n_entries"), ("refs", refs, "n_refs")):
        counted = frame.groupby(frame["arxiv_id"].astype(str)).size().reindex(p.index, fill_value=0)
        wrong = p.index[p[column].astype("int64").to_numpy() != counted.astype("int64").to_numpy()]
        if len(wrong):
            problems.append(f"pubs: {column} differs from the rows of {name} for {len(wrong)} papers, e.g. {list(wrong[:3])}")
    if (p["eligible"].astype(bool) != ((p["n_entries"] >= 1) & (p["n_refs"] >= 1))).any():
        problems.append("pubs: eligible is not (n_entries >= 1 and n_refs >= 1)")
    if len(entries):
        pos = entries.groupby(entries["arxiv_id"].astype(str))["position"].agg(["min", "max", "count", "nunique"])
        bad_pos = pos[(pos["min"] != 0) | (pos["max"] != pos["count"] - 1) | (pos["nunique"] != pos["count"])]
        if len(bad_pos):
            problems.append(f"entries: position is not 0..m-1 in {len(bad_pos)} papers, e.g. {list(bad_pos.index[:3])}")
        n_context = entries["n_context"] if "n_context" in entries else entries["context"].map(len)
        if ((entries["n_cites"] > 0) != (n_context > 0)).any():
            problems.append("entries: n_cites > 0 without a context, or a context with n_cites = 0")
        if (entries["n_cites"] < 0).any() or (n_context > CONTEXT_MAX).any():
            problems.append(f"entries: n_cites < 0 or more than {CONTEXT_MAX} context sentences")
    if parser_version:
        read = pubs[~pubs["m2_status"].isin(["missing", "unreadable"])]
        others = read["parser_version"].fillna("none")
        others = others[others != parser_version]
        if len(others):
            problems.append(f"pubs: {len(others)} parsed papers are not {parser_version}: "
                            f"{others.value_counts().head(3).to_dict()}")
    return problems


def _counts(series):
    return {str(k): int(v) for k, v in series.fillna("none").value_counts().items()}


def _in_pool(pubs, min_entries, min_refs, max_entries=None):
    mask = (pubs["n_entries"] >= min_entries) & (pubs["n_refs"] >= min_refs)
    return mask & (pubs["n_entries"] <= max_entries) if max_entries is not None else mask


def m3_summary(pubs, stats=None):
    """Numbers of data/m3/ for the group, the README and the poster (from `pubs` alone)."""
    p = pubs.assign(member=pubs["member"].fillna("none"))
    total = lambda column, frame=p: int(frame[column].astype("int64").sum())   # noqa: E731
    share = lambda a, b: round(a / b, 4) if b else None                         # noqa: E731
    eligible = p[p["eligible"].astype(bool)]
    by_member = {}
    for member, group in p.groupby("member", sort=True):
        pool = group[_in_pool(group, **MANUAL_POOL)]
        by_member[str(member)] = {
            "papers": len(group), "eligible": int(group["eligible"].astype(bool).sum()),
            "entries": total("n_entries", group), "arxiv_refs": total("n_refs", group),
            "manual_pool": len(pool), "manual_pool_bib": int(pool["bib_source"].eq("bib").sum()),
            "manual_pool_bibitem": int(pool["bib_source"].eq("bibitem").sum())}
    median = lambda column: float(eligible[column].median()) if len(eligible) else None   # noqa: E731
    return {
        "papers": len(p), "m2_status": _counts(p["m2_status"]), "s2_status": _counts(p["s2_status"]),
        "bib_source_of_papers_with_entries": _counts(p.loc[p["n_entries"] > 0, "bib_source"]),
        "papers_with_entries": int((p["n_entries"] > 0).sum()), "papers_with_arxiv_refs": int((p["n_refs"] > 0).sum()),
        "eligible": len(eligible), "eligible_share": share(len(eligible), len(p)),
        "entries": total("n_entries"), "entries_in_eligible": total("n_entries", eligible),
        "cited_entry_share": share(total("n_cited_entries"), total("n_entries")),
        "refs_all": total("n_refs_all"), "arxiv_refs": total("n_refs"),
        "arxiv_ref_share": share(total("n_refs"), total("n_refs_all")),
        "median_entries_per_eligible": median("n_entries"), "median_arxiv_refs_per_eligible": median("n_refs"),
        "candidate_pairs_in_eligible": int((eligible["n_entries"].astype("int64") * eligible["n_refs"].astype("int64")).sum()),
        "manual_pool": int(_in_pool(p, **MANUAL_POOL).sum()), "split_pool": int(_in_pool(p, **SPLIT_POOL).sum()),
        "parser_versions": _counts(p["parser_version"]), "by_member": by_member,
        "stats": {k: int(v) for k, v in (stats or {}).items()},
    }


def print_m3_summary(summary):
    s = summary
    pct = lambda x: f"{x:.1%}" if x is not None else "-"   # noqa: E731
    print(f"Papers {s['papers']:,} | with entries {s['papers_with_entries']:,} | with arXiv refs "
          f"{s['papers_with_arxiv_refs']:,} | eligible {s['eligible']:,} ({pct(s['eligible_share'])})")
    print(f"Entries {s['entries']:,} ({s['entries_in_eligible']:,} in eligible papers, cited in the text: "
          f"{pct(s['cited_entry_share'])}) | references {s['refs_all']:,}, with arXiv ID {s['arxiv_refs']:,} "
          f"({pct(s['arxiv_ref_share'])})")
    print(f"Eligible papers: median {s['median_entries_per_eligible']} entries, {s['median_arxiv_refs_per_eligible']} "
          f"arXiv refs, {s['candidate_pairs_in_eligible']:,} (entry, ref) pairs | manual pool {s['manual_pool']:,}, "
          f"split pool {s['split_pool']:,}")
    print(f"M2: {s['m2_status']} | S2: {s['s2_status']} | bib_source: {s['bib_source_of_papers_with_entries']}")
    print(f"Parser versions: {s['parser_versions']} | build stats: {s['stats']}")
    print(pd.DataFrame(s["by_member"]).T.to_string())

## 6. Tests

Offline tests on a small paper built in memory (two versions, a sentence kept in both and placed under two
sections, `\citep[..][..]{...}`, `\Citep`, `\nocite{*}`, a key cited but not in the bibliography, a repeated
key) and its `references.json` (a reference without arXiv ID, an old-style ID, an `arXiv:...v2` ID, a
reference listed twice). With Parquet available (Colab), `build_m3_data` runs end to end on a temporary
folder. Two checks use real files when they exist: `ref_rows` against `references_rows` of `03` on 40 papers
of `data/raw/`, and the rows of 20 papers of `data/parsed/` against `check_m3_tables`.

In [ ]:
_S1 = r"We extend \cite{a}."
_S2 = r"Prior work \cite{a, b} and \citep[see][p.~2]{c}."
_S3 = r"Following \citet{b}, we use \Citep{b}."


def _fixture_doc(arxiv_id="2609.32834", versions=None, keys=("a", "b", "c", "e", "a")):
    """A parsed.json document like 15 writes: v1 and v2, a sentence (s2) in both and under two sections."""
    f = to_folder_id(arxiv_id)
    i = lambda name: f"{f}#{name}"   # noqa: E731
    elements = [("doc", "document", "A paper"), ("abs1", "abstract", "Abstract"), ("s1", "sentence", _S1),
                ("sec1", "section", "Introduction"), ("s2", "sentence", _S2), ("ssec1", "subsection", "Setup"),
                ("s3", "sentence", _S3), ("sec2", "section", "Related Work"), ("s4", "sentence", "No citation."),
                ("eq1", "formula", r"\begin{equation}x=1\end{equation}"),
                ("s5", "sentence", r"New in v2 \cite{d}.\nocite{*}")]
    trees = {"1": [i("doc"), [i("abs1"), [i("s1")]], [i("sec1"), [i("s2")], [i("ssec1"), [i("s3")]]],
                   [i("sec2"), [i("s2")], [i("s4")], [i("eq1")]]],
             "2": [i("doc"), [i("sec1"), [i("s2")], [i("s5")]]]}
    bib = [{"arxiv_id": arxiv_id, "citation_key": k, "entry_type": "article", "title": f"Title {k}",
            "authors": ["Ann Smith", "Bo Li"], "year": 2020, "doi": None, "journal": None, "url": None} for k in keys]
    return {"parser_version": M3_PARSER_VERSION, "arxiv_id": arxiv_id, "folder_id": f,
            "versions": versions or [{"version": 1, "status": "ok", "bib_source": "bibitem"},
                                     {"version": 2, "status": "ok", "bib_source": "bib"}],
            "counts": {}, "trees": trees,
            "elements": {"element_id": [i(e) for e, _, _ in elements], "arxiv_id": [arxiv_id] * len(elements),
                         "kind": [k for _, k, _ in elements], "content": [c for _, _, c in elements]},
            "hierarchy": {c: [] for c in HIERARCHY_COLUMNS},
            "bibtex": {c: [row[c] for row in bib] for c in BIBTEX_COLUMNS}}


def _fixture_refs(arxiv_id="2609.32834", status="ok"):
    refs = [
        {"s2_id": "s0", "title": "Zero", "authors": ["A B"], "submission_date": "2025-04-08", "year": 2025,
         "venue": None, "arxiv_id": "2504.06381", "external_ids": {"ArXiv": "2504.06381"}},
        {"s2_id": "s1", "title": "Not on arXiv", "authors": [], "submission_date": "2024", "year": 2024,
         "venue": "J", "arxiv_id": None, "external_ids": {"DOI": "10.1/x"}},
        {"s2_id": "s2", "title": "Old style", "authors": ["C D"], "submission_date": "1999-01", "year": None,
         "venue": "Phys. Rev. D", "arxiv_id": "hep-th/9901001",
         "external_ids": {"ArXiv": "hep-th/9901001", "DOI": "10.1103/PhysRevD.1"}},
        {"s2_id": "s3", "title": "Zero again", "authors": ["A B"], "submission_date": "2025", "year": 2025,
         "venue": None, "arxiv_id": "2504.06381", "external_ids": {"ArXiv": "2504.06381"}},
        {"s2_id": "s4", "title": "Versioned", "authors": ["E F"], "submission_date": "2023-10-01", "year": 2023,
         "venue": "", "arxiv_id": "arXiv:2310.12345v2", "external_ids": {"ArXiv": "2310.12345"}},
    ]
    return {"id": arxiv_id, "status": status, "n_refs": len(refs), "references": refs if status == "ok" else []}


class M3DataTests(unittest.TestCase):
    PAPERS = [("2603.20580", "2603-20580"), ("2605.07338", "2605-07338"), ("2605.17338", "2605-17338"),
              ("2609.32834", "2609-32834")]

    @classmethod
    def setUpClass(cls):
        """A project folder: 2609.32834 complete; 2603.20580 without parsed file; 2605.07338 not found by S2;
        2605.17338 with a torn parsed file."""
        cls.tmp = Path(tempfile.mkdtemp(prefix="m3-test-"))
        cls.parsed, cls.raw = cls.tmp / "parsed", cls.tmp / "raw"
        cls.parsed.mkdir()
        for arxiv_id, folder in cls.PAPERS:
            (cls.raw / folder).mkdir(parents=True)
            status = "not_found" if arxiv_id == "2605.07338" else "ok"
            write_json(cls.raw / folder / "references.json", _fixture_refs(arxiv_id, status))
        write_json(cls.parsed / "2609-32834.json", _fixture_doc("2609.32834"), indent=None)
        write_json(cls.parsed / "2605-07338.json", _fixture_doc("2605.07338"), indent=None)
        (cls.parsed / "2605-17338.json").write_text('{"parser_version": "13.8', encoding="utf-8")

    @classmethod
    def tearDownClass(cls):
        shutil.rmtree(cls.tmp, ignore_errors=True)

    def _frames(self):
        e, r, p, stats = rows_for_papers(self.PAPERS, self.parsed, self.raw, versions={}, threads=2)
        return (*frames_of(e, r, p), stats)

    def test_contexts_counted_once_with_every_section(self):
        found = citation_contexts(_fixture_doc())
        self.assertEqual(found["a"], {"context": [_S1, _S2], "n_cites": 2,
                                      "sections": ["Abstract", "Introduction", "Related Work"]})
        self.assertEqual(found["b"], {"context": [_S2, _S3], "n_cites": 3,   # s3 is in a subsection of sec1
                                      "sections": ["Introduction", "Related Work"]})
        self.assertEqual(found["c"], {"context": [_S2], "n_cites": 1, "sections": ["Introduction", "Related Work"]})
        self.assertEqual(set(found), {"a", "b", "c", "d", "*"})

    def test_context_cap_keeps_the_count(self):
        found = citation_contexts(_fixture_doc(), max_sentences=1)
        self.assertEqual(found["a"]["context"], [_S1])
        self.assertEqual(found["a"]["n_cites"], 2)

    def test_entry_rows(self):
        rows, stats = entry_rows("2609.32834", _fixture_doc())
        self.assertEqual([r["citation_key"] for r in rows], ["a", "b", "c", "e"])
        self.assertEqual([r["position"] for r in rows], [0, 1, 2, 3])
        self.assertEqual([r["n_cites"] for r in rows], [2, 3, 1, 0])
        self.assertEqual(rows[3]["context"], [])
        self.assertEqual(rows[3]["sections"], [])
        self.assertTrue(all(r["bib_source"] == "bib" for r in rows))
        self.assertEqual(stats["entry_duplicate_keys"], 1)
        self.assertEqual(stats["cited_keys_not_in_bibtex"], 1)   # d; \nocite{*} is not a key
        self.assertEqual(set(rows[0]), set(ENTRY_COLUMNS))

    def test_bib_source_of_a_paper(self):
        self.assertEqual(paper_bib_source({"versions": [{"status": "ok", "bib_source": "bib"},
                                                        {"status": "empty_source", "bib_source": "none"}]}), "bib")
        self.assertEqual(paper_bib_source({"versions": [{"status": "ok", "bib_source": "missing_bbl"}]}), "missing_bbl")
        self.assertEqual(paper_bib_source({"versions": []}), "none")
        self.assertEqual(paper_bib_source(None), "none")

    def test_ref_rows(self):
        rows, dropped = ref_rows("2609.32834", _fixture_refs())
        self.assertEqual([r["ref_arxiv_id"] for r in rows], ["2504.06381", "hep-th/9901001", "2310.12345"])
        self.assertEqual([r["ref_position"] for r in rows], [0, 2, 4])
        self.assertEqual(dropped, 1)
        self.assertEqual(rows[1]["ref_doi"], "10.1103/PhysRevD.1")
        self.assertIsNone(rows[1]["ref_year"])
        self.assertIsNone(rows[2]["ref_venue"])
        self.assertEqual(set(rows[0]), set(REF_COLUMNS))
        self.assertEqual(ref_rows("2609.32834", _fixture_refs(status="not_found")), ([], 0))
        self.assertEqual(ref_rows("2609.32834", None), ([], 0))

    def test_member_of(self):
        cases = {"2603.20580": "TV1", "2603.25579": "TV1", "2603.25580": None, "2609.32834": "TV2",
                 "2609.37833": "TV2", "2609.37834": None, "2609.32834v2": "TV2", "2605.17338": "TV3",
                 "2605.22337": "TV3", "2605.07338": "TV4", "2605.12337": "TV4", "2605.17337": None,
                 "hep-th/9901001": None}
        self.assertEqual({i: member_of(i) for i in cases}, cases)

    def test_spread_sample(self):
        papers = [(f"{yymm}.{n:05d}", f"{yymm}-{n:05d}") for yymm, first in
                  (("2603", 20580), ("2605", 7338), ("2605", 17338), ("2609", 32834)) for n in range(first, first + 100)]
        sample = spread_sample(papers, 8)
        self.assertEqual(len(sample), 8)
        self.assertEqual(Counter(member_of(a) for a, _ in sample), {"TV1": 2, "TV2": 2, "TV3": 2, "TV4": 2})
        self.assertEqual(sample, sorted(sample, key=_paper_order))

    def test_rows_of_papers_pass_the_checks(self):
        entries, refs, pubs, stats = self._frames()
        self.assertEqual(check_m3_tables(entries, refs, pubs), [])
        p = pubs.set_index("arxiv_id")
        self.assertEqual(p.loc["2609.32834", ["member", "n_entries", "n_refs", "eligible", "n_cited_entries",
                                             "n_refs_all", "m2_status", "s2_status"]].tolist(),
                         ["TV2", 4, 3, True, 3, 5, "ok", "ok"])
        self.assertEqual(p.loc["2603.20580", ["n_entries", "n_refs", "eligible", "m2_status"]].tolist(),
                         [0, 3, False, "missing"])
        self.assertEqual(p.loc["2605.07338", ["n_entries", "n_refs", "eligible", "s2_status", "n_refs_all"]].tolist(),
                         [4, 0, False, "not_found", 0])
        self.assertEqual(p.loc["2605.17338", ["n_entries", "m2_status"]].tolist(), [0, "unreadable"])
        self.assertEqual(stats["ref_duplicates_dropped"], 3)
        self.assertEqual(list(entries["arxiv_id"].unique()), ["2605.07338", "2609.32834"])

    def test_checks_find_problems(self):
        entries, refs, pubs, _ = self._frames()
        problems = check_m3_tables(pd.concat([entries, entries.iloc[:1]], ignore_index=True), refs, pubs)
        self.assertTrue(any("duplicate (arxiv_id, citation_key)" in p for p in problems), problems)
        self.assertTrue(any("n_entries differs" in p for p in problems), problems)
        wrong = pubs.copy()
        wrong.loc[wrong["arxiv_id"] == "2609.32834", "parser_version"] = "13.7|10:team|11:team|12:team"
        self.assertTrue(any("are not 13.9" in p for p in check_m3_tables(entries, refs, wrong)))
        self.assertEqual(check_m3_tables(entries, refs, wrong, parser_version=None), [])
        self.assertTrue(check_m3_tables(entries.drop(columns="sections"), refs, pubs))

    def test_summary(self):
        entries, refs, pubs, stats = self._frames()
        s = m3_summary(pubs, stats)
        self.assertEqual((s["papers"], s["eligible"], s["entries"], s["arxiv_refs"]), (4, 1, 8, 9))
        self.assertEqual(s["candidate_pairs_in_eligible"], 12)
        self.assertEqual(s["by_member"]["TV2"]["eligible"], 1)
        json.dumps(s)   # serializable as it is

    def test_arrow_tables_have_the_contract_types(self):
        e, r, p, _ = rows_for_papers(self.PAPERS, self.parsed, self.raw, versions={}, threads=2)
        for table, rows in zip(M3_TABLES, (e, r, p)):
            t = _to_table(rows, M3_SCHEMAS[table])
            self.assertEqual(t.num_rows, len(rows))
            self.assertEqual(tuple(t.to_pandas().columns), tuple(M3_SCHEMAS[table].names))

    def _needs_parquet(self):
        try:
            import pyarrow.parquet  # noqa: F401
        except ImportError as err:
            self.skipTest(f"pyarrow.parquet not available here ({str(err)[:60]}...)")

    def test_parser_index_and_the_check_before_a_build(self):
        index = self.tmp / "index.jsonl"
        lines = [{"folder_id": "2609-32834", "parser_version": M3_PARSER_VERSION},
                 {"folder_id": "2605-07338", "parser_version": M3_PARSER_VERSION},
                 {"folder_id": "2605-07338", "parser_version": None},   # unreadable in 15: parsed again later
                 {"folder_id": "2609-32834", "parser_version": "13.7|10:team|11:team|12:team"}]
        index.write_text("\n".join(map(json.dumps, lines)) + '\n{"folder_id": "2603-2', encoding="utf-8")
        self.assertEqual(m2_parser_versions(index), {"2609-32834": "13.7|10:team|11:team|12:team", "2605-07338": None})
        self._needs_parquet()
        with self.assertRaisesRegex(RuntimeError, "1 papers are not 13.9"):
            build_m3_data(self.PAPERS, out_dir=self.tmp / "m3_other", parsed_dir=self.parsed, raw_dir=self.raw,
                          index=index, force=True)
        self.assertFalse((self.tmp / "m3_other").exists())   # stopped before writing anything

    def test_build_end_to_end(self):
        self._needs_parquet()
        out = self.tmp / "m3"
        record = build_m3_data(self.PAPERS, out_dir=out, batch=2, threads=2, force=True, parsed_dir=self.parsed,
                               raw_dir=self.raw, index=self.tmp / "no_index.jsonl")
        self.assertEqual(record["rows"], {"entries": 8, "refs": 9, "pubs": 4})
        self.assertFalse((out / "_parts").exists())
        frames = load_m3(out_dir=out)
        self.assertEqual(check_m3_tables(frames["entries"], frames["refs"], frames["pubs"]), [])
        e = frames["entries"]
        row = e[(e["arxiv_id"] == "2609.32834") & (e["citation_key"] == "a")].iloc[0]
        self.assertEqual(list(row["context"]), [_S1, _S2])
        self.assertEqual(list(row["sections"]), ["Abstract", "Introduction", "Related Work"])
        self.assertEqual(str(e["entry_type"].dtype), "category")   # dictionary-encoded, like M2's bibtex
        again = build_m3_data(self.PAPERS, out_dir=out, batch=2, threads=2, parsed_dir=self.parsed, raw_dir=self.raw,
                              index=self.tmp / "no_index.jsonl", force=False)
        self.assertEqual(again["signature"], record["signature"])

    def test_ref_rows_match_references_rows_of_03(self):
        sample = [(a, f) for a, f in m3_papers()[:200] if (DATA_DIR / f / "references.json").is_file()][:40]
        if not sample:
            self.skipTest("no references.json in data/raw/ here")
        for arxiv_id, folder in sample:
            ours = [{c: r[c] for c in REF_COLUMNS[:6]} for r in ref_rows(arxiv_id, read_json(DATA_DIR / folder / "references.json"))[0]]
            theirs, seen = [], set()
            for row in H03["references_rows"]([arxiv_id]):
                if row["ref_arxiv_id"] not in seen:
                    seen.add(row["ref_arxiv_id"])
                    theirs.append(row)
            self.assertEqual(ours, theirs, arxiv_id)

    def test_real_papers(self):
        sample = [p for p in spread_sample(m3_papers(), 40) if (PARSED_DIR / f"{p[1]}.json").is_file()][:20]
        if not sample:
            self.skipTest("no data/parsed/ here")
        e, r, p, stats = rows_for_papers(sample)
        entries, refs, pubs = frames_of(e, r, p)
        self.assertEqual(check_m3_tables(entries, refs, pubs, parser_version=None), [])
        print(f"\n[20] {len(sample)} real papers: {len(entries)} entries (cited in the text: "
              f"{(entries['n_cites'] > 0).mean():.0%}), {len(refs)} arXiv refs, {int(pubs['eligible'].sum())} "
              f"eligible | {dict(stats)}")


if RUN_TEST:
    _result = unittest.TextTestRunner(verbosity=1).run(unittest.TestLoader().loadTestsFromTestCase(M3DataTests))
    assert _result.wasSuccessful(), "20: tests failed"

## 7. Run (Colab)

With `BUILD_M3 = True`: every paper of `m3_papers()` (or `TEST_PAPERS` of them, into `data/m3_test/`), then
the summary for the group. With `PREPARE_SHEETS = True` as well (real build only), `21_labeling` then picks
the 5 publications of each member (`labels/selection.csv`, never overwritten afterwards) and writes their
empty sheets to `labels/todo/<member>/`, in the same session: Drive is unmounted only at the very end.
After the real build, send the summary to the group: the parser is locked and manual labelling can start.

In [ ]:
if BUILD_M3:
    _out = M3_DIR if not TEST_PAPERS else M3_DIR.parent / "m3_test"
    M3_BUILD = build_m3_data(m3_papers(TEST_PAPERS), out_dir=_out)
    print_m3_summary(M3_BUILD["summary"])
    if PREPARE_SHEETS and not TEST_PAPERS:
        get_ipython().run_line_magic("run", "-i ./21_labeling.ipynb")
        M3_SELECTION = prepare_manual_sheets()
        print(M3_SELECTION.to_string())
    if ON_COLAB and FLUSH_DRIVE_AT_END:
        from google.colab import drive
        drive.flush_and_unmount()
        print("Drive flushed and unmounted: every file of data/m3/ is saved")

In [ ]:
print(f"20 loaded | parser locked: {M3_PARSER_VERSION} | data version {M3_DATA_VERSION} | M3_DIR: {M3_DIR}")